# 1. Import the necessary libraries

In [1]:
from mesh.core import *
from mesh.parameters import MeshParameters
from microgrid_old.techno_ka import techno_ka
from problems import get_problem
from runners import run_mesh, run_mesh_old, run_nsga2

from functools import partial
from concurrent.futures import ProcessPoolExecutor, as_completed

import contextlib
import io
import numpy as np
import os

# 2. Define the general configuration of the experiments

In [2]:
#################### CUSTOMIZABLE ####################

# Number of processes to execute the experiments in parallel
workers = 6

# Objective function configuration
experiment_names = ['zdt1', 'zdt2', 'zdt3']
objective_dim = 2 # Number of objectives
position_dim = 10 # Number of decision variables

# Execution configuration
num_runs = 1 # Number of runs
max_fitness_eval = 3000 # Maximum fitness evaluations (not used if it is None)
population_size = 100 # Population size
random_state = None # Set a seed for random numbers (not used if it is None)

results_folder = f'results' # Folder to store the results
tuning_folder = f'hyperparams' # Folder to get the tuned parameters
######################################################

# Function to capture errors without stopping the other processes
def safe_run(run_function, params):
	try:
		f = io.StringIO()
		with contextlib.redirect_stdout(f):
			status = run_function(*params)
		return status, f.getvalue()
	except Exception as e:
		return f'Error: {str(e)}'

# 3 Define functions to set up the fitness function

In [3]:
# Microgrid objetive function
# position_min_value = np.array([10, 1, 50]) # Lower bound of problem [max PV generation, number of wind turbines, battery capacity]
# position_max_value = np.array([450, 5, 500]) # Upper bound of problem [max PV generation, number of wind turbines, battery capacity]
# def func(args):
#     r = techno_ka(args[0], args[1], 0.8, args[2], select_bat, solar_data, wind_data, load_ind)[:objective_dim]
#     #r = techno_ka(args[0], args[1], 0.8, args[2], select_bat, solar_data, wind_data, load_ind)[1:3]
#     r[-1] = -r[-1] # Maximizing renewable factor
#     return r

def get_fitness_function(func_name, objective_dim, position_dim):
	func, position_min_value, position_max_value = get_problem(func_name, n_var=position_dim, n_obj=objective_dim)
	return func, objective_dim, position_dim, position_min_value, position_max_value

# 3. Run experiments with the algorithms (in parallel)

## 3.1 Run MESH

In [ ]:
#################### CUSTOMIZABLE ####################

# MESH fixed parameters
memory_size = population_size # Maximum number of particles in memory

mesh_global_best_type = [0,1] # 0 -> Sigma method (G1) | 1 -> Sigma method in fronts (G2)
mesh_dm_pool_type = [0] # 0 -> Sampling from memory (S1) | 1 -> Sampling from population (S2) | 2 -> Sampling from memory and population (S3)
mesh_differential_evolution_type = [0] # 0 -> DE\rand\1\Bin (D1) | 1 -> DE\rand\2\Bin (D2) | 2 -> DE/Best/1/Bin (D3) | 3 -> DE/Current-to-best/1/Bin (D4) | 4 -> DE/Current-to-rand/1/Bin (D5)

# MESH tunable parameters
# OBS: The function "run_mesh" and "run_mesh_old" will select automatically the tuned parameters if they exists ("hyperparams" folder generated by "fine_tuning.ipynb")
communication_probability = 0.7 # Communication probability
mutation_rate = 0.4 # Mutation rate
personal_guide_array_size = 3 # Number of personal guides
######################################################

In [ ]:
# Set the list of parameters
params_list = [
  	[(F'MESH_G{gb_type+1}S{pool_type+1}D{de_type+1}_{exp_name}', results_folder, tuning_folder, num_runs, max_fitness_eval, population_size, random_state),
	 get_fitness_function(exp_name, objective_dim, position_dim),
	 (memory_size, gb_type, pool_type, de_type),
	 (communication_probability, mutation_rate, personal_guide_array_size)]
     for exp_name in experiment_names
     for gb_type in mesh_global_best_type
	 for pool_type in mesh_dm_pool_type
	 for de_type in mesh_differential_evolution_type
]

# Execute in parallel
Parallel(n_jobs=workers)(delayed(safe_run)(run_mesh, params) for params in params_list)

[('MESH_G1S1D1_zdt1 was successfully executed!', 'cp = 0.5 mr = 0.4 pgas = 3'),
 ('MESH_G2S1D1_zdt1 was successfully executed!', 'cp = 0.7 mr = 0.4 pgas = 3'),
 ('MESH_G1S1D1_zdt2 was successfully executed!', 'cp = 0.7 mr = 0.4 pgas = 3'),
 ('MESH_G2S1D1_zdt2 was successfully executed!', 'cp = 0.7 mr = 0.4 pgas = 3'),
 ('MESH_G1S1D1_zdt3 was successfully executed!', 'cp = 0.7 mr = 0.4 pgas = 3'),
 ('MESH_G2S1D1_zdt3 was successfully executed!', 'cp = 0.7 mr = 0.4 pgas = 3')]

### Run MESH (previous implementation)

In [ ]:
# Set the list of parameters
params_list = [
  	[(F'OLD_MESH_G{gb_type+1}S{pool_type+1}D{de_type+1}_{exp_name}', results_folder, tuning_folder, num_runs, max_fitness_eval, population_size, random_state),
	 get_fitness_function(exp_name, objective_dim, position_dim),
	 (memory_size, gb_type, pool_type, de_type),
	 (communication_probability, mutation_rate, personal_guide_array_size)]
     for exp_name in experiment_names
     for gb_type in mesh_global_best_type
	 for pool_type in mesh_dm_pool_type
	 for de_type in mesh_differential_evolution_type
]

# Execute in parallel
Parallel(n_jobs=workers)(delayed(safe_run)(run_mesh_old, params) for params in params_list)

[('OLD_MESH_G1S1D1_zdt1 was successfully executed!', ''),
 ('OLD_MESH_G2S1D1_zdt1 was successfully executed!', ''),
 ('OLD_MESH_G1S1D1_zdt2 was successfully executed!', ''),
 ('OLD_MESH_G2S1D1_zdt2 was successfully executed!', ''),
 ('OLD_MESH_G1S1D1_zdt3 was successfully executed!', ''),
 ('OLD_MESH_G2S1D1_zdt3 was successfully executed!', '')]

# 3.2 Run NSGA-II

In [4]:
#################### CUSTOMIZABLE ####################

# NSGA2 tunable parameters
# OBS: The function "run_nsga2" will select automatically the tuned parameters if they exists ("hyperparams" folder generated by "fine_tuning.ipynb")
recombination_probability = 0.45
eta_recombination = 19
mutation_probability = 0.06
eta_mutation = 14.7
######################################################

In [5]:
# Set the list of parameters
params_list = [
  	[(F'NSGA2_{exp_name}', results_folder, tuning_folder, num_runs, max_fitness_eval, population_size, random_state),
	 get_fitness_function(exp_name, objective_dim, position_dim),
	 (),
	 (recombination_probability, eta_recombination, mutation_probability, eta_mutation)]
     for exp_name in experiment_names
]

# Execute in parallel
Parallel(n_jobs=workers)(delayed(safe_run)(run_nsga2, params) for params in params_list)

[('NSGA2_zdt1 was successfully executed!', ''),
 ('NSGA2_zdt2 was successfully executed!', ''),
 ('NSGA2_zdt3 was successfully executed!', '')]

## 3.3 Run ...